# Supervised Learning 2

## Learning Objectives
* Being able to measure the [performance](https://janalasser.at/lectures/MD_KI/VO2_2_performance/) of a classifier on test data.
* Being able to implement supervised learning for [regression](https://janalasser.at/lectures/MD_KI/VO2_5_regression/) in Python.
* Being able to measure the performance of a regression.

## Train-Test Procedure
We again use the [K-Nearest Neighbors](https://janalasser.at/lectures/MD_KI/VO1_3_algorithms_supervised_learning/#/2/0/9) algorithm, which we also used in the last part of the course. In the previous session, we tried hyperparameters (e.g., the number of neighbors to include in the classification) more or less at random. Now we want to approach this more systematically.

In [ ]:
# as a first step, we import the dataset we want to work with
import seaborn as sns
titanic_data = sns.load_dataset("titanic")

# filter the data and remove NaN values
selected_columns = ["age", "fare", "pclass", "adult_male", "survived"]
filtered_titanic_data = titanic_data[selected_columns].copy()
filtered_titanic_data = filtered_titanic_data.dropna()

# define which columns contain the features and which
# column is the label or target value we want to predict
label = "survived"
features = ["age", "fare", "pclass", "adult_male"]

x = filtered_titanic_data[features]
y = filtered_titanic_data[label]

In [ ]:
# and train a classifier with n_neighbors=5
from sklearn.neighbors import KNeighborsClassifier

knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(x, y)

In the [lecture](https://janalasser.at/lectures/MD_KI/VO2_2_performance/#/1/1/1), we learned that one way to measure the performance of a classifier is accuracy. It is defined as 1 minus the classifier’s error rate. In `scikit-learn`, we can calculate accuracy using the `score()` function of a trained classifier.

The `score()` function takes as arguments the features and target values of the observations, predicts the target values for each observation based on the features, and then compares the predictions with the true target values to calculate the error rate or accuracy.

In [ ]:
# calculate the accuracy
knn_classifier.score(x, y)

In the [lecture](https://janalasser.at/lectures/MD_KI/VO2_3_bias_variance/#/1/3/1), we also discussed that classifiers tend to “memorize” their training data (overfitting). Accordingly, the prediction error we calculated above using the training data likely underestimates the true prediction error for data the classifier has not seen during training.

Therefore, we split the dataset into a part used for training (training data) and a part used to measure performance (test data).

<div>
<img src="https://drive.google.com/uc?id=1EK5CUuDWoFDdc7vl6A6tnamNR4K1edcm" width="700"/>
</div>

`scikit-learn` provides a built-in function `train_test_split()` that handles splitting the data for us. It returns four DataFrames. Which observations end up in which dataset is determined randomly:

* `x_training`: the features of the training data
* `x_test`: the features of the test data
* `y_training`: the target values (labels) of the training data
* `y_test`: the target values (labels) of the test data

The `test_size` parameter specifies the proportion of the data to use as test data – here 20%.

The `random_state` parameter fixes the randomness. That means if we call `train_test_split()` twice with the same `random_state`, it will return the same split of training and test data. It is good practice to always set `random_state` to some (arbitrary) value so that training results and performance measurements are reproducible and not dependent on chance.

In [ ]:
from sklearn.model_selection import train_test_split
x_training, x_test, y_training, y_test = train_test_split(x, y, test_size=0.2, random_state=66)

# we train our classifier again, this time using only the training data
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(x_training, y_training)

In [ ]:
# and we evaluate the performance, but this time using the test data
knn_classifier.score(x_test, y_test)

In [ ]:
# for comparison: the performance on the training data is noticeably higher
knn_classifier.score(x_training, y_training)

## Systematically Evaluating Classification Performance
We again use the [K-Nearest Neighbors](https://janalasser.at/lectures/MD_KI/VO1_3_algorithms_supervised_learning/#/2/0/9) algorithm, which we also used in the last part of the course. Previously, we tried hyperparameters (e.g., the number of neighbors to include in the classification) more or less at random. Now we want to approach this in a more systematic way.

In [ ]:
accuracy = []  # list to store the achieved accuracy
current_neighbors = []  # list to store the current n_neighbors
mode = []  # "training" or "test"

# try n_neighbors from 1 to 35
neighbor_range = range(1, 36)

for n_neighbors in neighbor_range:
    # train the classifier
    knn_classifier = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn_classifier.fit(x_training, y_training)

    # calculate accuracy on the training data
    training_accuracy = knn_classifier.score(x_training, y_training)

    # store the result, number of neighbors, and mode in lists
    accuracy.append(training_accuracy)
    current_neighbors.append(n_neighbors)
    mode.append("training")

    # calculate accuracy on the test data
    test_accuracy = knn_classifier.score(x_test, y_test)

    # store the result, number of neighbors, and mode in lists
    accuracy.append(test_accuracy)
    current_neighbors.append(n_neighbors)
    mode.append("test")

In [ ]:
# finally, we store the results in a DataFrame
import pandas as pd
# create an empty DataFrame
accuracy_data = pd.DataFrame()
# add the accuracy results as a column
accuracy_data["accuracy"] = accuracy
# add a column with the number of neighbors
accuracy_data["n_neighbors"] = current_neighbors
# add a column with the mode ("training" or "test")
accuracy_data["mode"] = mode
# display the first rows of the DataFrame
accuracy_data.head()

In [ ]:
# visualize accuracy for different values of n_neighbors
# for training and test data
sns.lineplot(accuracy_data, x="n_neighbors", y="accuracy", hue="mode")

<font color='blue'><b>Exercise 1</b></font>  
<font color='blue'>In the code cell below, a dataset of features from breast images is loaded, indicating whether they contain breast cancer or are cancer-free. For more information, you can check the [dataset descriptor](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) and the related [research article](https://minds.wisconsin.edu/bitstream/handle/1793/59692/TR1131.pdf;jsessionid=B20185806547CEC53EE4CFC609B11264?sequence=1). The dataset is already split into training and test sets in the code cell.</font>

<ul class="outside">
<li><font color='blue'>Familiarize yourself with the dataset: how many observations are there? How many features? How often was breast cancer diagnosed?</font></li>
<li><font color='blue'>Train a K-Nearest Neighbors classifier with <tt>n_neighbors=11</tt> on the training data. What accuracy does the classifier achieve on the test data?</font></li>
<li><font color='blue'>Does the accuracy improve if, instead of the default setting where each observation is weighted equally, you use distance-based weighting (<tt>weights="distance"</tt>)?</font></li>
<li><font color='blue'>Try all values of <tt>n_neighbors</tt> between 1 and 30 and visualize the result as above. Which value gives the best performance on the test data?</font></li>
</ul>

In [ ]:
# load breast cancer data
from sklearn import datasets
breast_cancer = datasets.load_breast_cancer(as_frame=True)
# split data into features and target
X = breast_cancer["data"]
y = breast_cancer["target"]
# split data into training and test sets
X_training, X_test, y_training, y_test = train_test_split(X, y, test_size=0.2, random_state=66)

In [ ]:
# Your code here

## Regression
In addition to classification, [regression](https://janalasser.at/lectures/MD_KI/VO2_5_regression/) (predicting a numerical value) is the second major application area of supervised learning. Implementing a regression algorithm in Python is very similar to implementing a classifier.

Below, we illustrate this using a sample dataset of diabetes patients. We are interested in predicting the progression of diabetes based on indicators such as age, sex, BMI, and blood measurements.

In [ ]:
diabetes = datasets.load_diabetes(as_frame=True)
x = diabetes["data"]
y = diabetes["target"]

In [ ]:
# 10 features: age, sex, BMI, blood pressure, and six blood serum measurements
# (tc, ldl, hdl, tch, ltg, glu) from 442 diabetes patients.
# Preprocessing: the values were centered (mean subtracted) and scaled by
# the standard deviation * sqrt(number of observations)
x.head()

In [ ]:
sns.histplot(data=x, x="age")

In [ ]:
sns.histplot(data=x, x="sex")

In [ ]:
# Target value: a measure of diabetes progression after one year
y.head()

In [ ]:
# We visualize diabetes progression versus blood pressure "bp"

# create a DataFrame containing both the features and the target value as columns
plot_data = diabetes["data"].copy()
plot_data["progress"] = diabetes["target"]

# use seaborn's scatterplot() function to visualize the data
# there appears to be a relationship between blood pressure and diabetes progression
sns.scatterplot(plot_data, x="bp", y="progress")

We now want to find a function that best describes the relationship between features and the target value. Here, we restrict ourselves to linear functions (linear regression).

<div>
<img src="https://drive.google.com/uc?id=1HaS6cI8w4HqfIKgAWACohwsfIrLnIzNv" width="400"/>
</div>

In [ ]:
# for illustration purposes, we first restrict ourselves to a single
# feature (blood pressure, "bp")
interesting_columns = ["bp"]
X = diabetes["data"][interesting_columns]

In [ ]:
# split the data into training and test sets
x_training, x_test, y_training, y_test = train_test_split(x, y, test_size=0.2, random_state=66)

# import the linear regressor from scikit-learn
from sklearn.linear_model import LinearRegression
linear_regressor = LinearRegression()
# train the model using the training data
linear_regressor.fit(x_training, y_training)

In [ ]:
# predict the diabetes progression for the test data
predictions = linear_regressor.predict(x_test)

In [ ]:
# create a DataFrame containing the features, the target value,
# and the predictions as columns
plot_data = x_test.copy()
plot_data["progress"] = y_test
plot_data["prediction"] = predictions

# visualize the observations and the predictions
sns.scatterplot(plot_data, x="bp", y="progress", color="black", label="observations")
sns.lineplot(plot_data, x="bp", y="prediction", color="orange", label="prediction");

## Measuring Regression Performance
As discussed in the [lecture](https://janalasser.at/lectures/MD_KI/VO2_5_regression/#/2/1/3), for regression we cannot rely on error classes like "false positive" or "false negative" to measure the model's error rate or performance. Instead, we measure the deviation of the true observations from the predictions, typically using the root mean squared error (RMSE).

<div>
<img src="https://drive.google.com/uc?id=1A9ienaKBg_vPtJYWuIRwS2bXYLxm2f-M" width="400"/>
</div>

In [ ]:
# import the function for mean squared error from scikit-learn
from sklearn.metrics import mean_squared_error
# import the square root function from the math module
from math import sqrt

# calculate the square root of the mean squared error for the test data
MSE = mean_squared_error(y_test, predictions)
RMSE = sqrt(MSE)
print(f"Root of the mean squared error (test data): {RMSE}")

For comparison, we also want to calculate the performance on the training data:

In [ ]:
# predict diabetes progression for the training data
predictions_training = linear_regressor.predict(x_training)

# calculate the square root of the mean squared error
MSE_training = mean_squared_error(y_training, predictions_training)
RMSE_training = sqrt(MSE_training)
print(f"Root of the mean squared error (training data): {RMSE_training}")

In [ ]:
# alternatively, the agreement between predictions and true values can also
# be assessed using the (Pearson) correlation coefficient
plot_data[["progress", "prediction"]].corr(method="pearson")

<font color='blue'><b>Exercise 2</b></font> <font color='blue'>For illustration, we previously used only the blood pressure column "bp" to predict diabetes progression. Train the model again, this time using all available features. Does the mean squared error improve?</font>

In [ ]:
# Your code here

## Homeworks

The homework for this course section can be found in [this notebook](https://colab.research.google.com/drive/1hEHtlMQh794Lv8Th6J0VFksVBDFbbwm8?usp=sharing).

## Additional Materials

* **Machine Learning**: [Book](https://www.amazon.de/Introduction-Machine-Learning-Python-Scientists/dp/1449369413?shipTo=AT&source=ps-sl-shoppingads-lpcontext&ref_=fplfs&psc=1&smid=A3JWKAKR8XB7XF&language=de_DE&gQT=1) *Introduction to Machine Learning with Python: A Guide for Data Scientists* with extensive [examples and exercises](https://github.com/amueller/introduction_to_ml_with_python) in Python.

## Source and License

This notebook was created by Jana Lasser for Course "B1 - Technical Aspects" of the Microcredential "AI and Society" at the University of Graz.

The notebook may be used, modified, and redistributed under the terms of the [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0) license.

This notebook was translated from German using GPT-5 and cross-checked by Alina Herderich.